# 13. Digital Filter Design, Cutoff Optimization & Causal Validation

**Objective**: Design Butterworth low-pass filters, sweep cutoff frequencies (0.5 to 3.0 Hz), and evaluate causal real-time streaming.

## 1. Setup & Candidate Filters

In [ ]:
import sys
sys.path.insert(0, '../..')
import numpy as np
import matplotlib.pyplot as plt
from Data_details.src.filter_design import design_butterworth_lowpass, apply_butterworth_causal, apply_butterworth_offline, RealTimeCausalFilter
from Data_details.src.dataset_loader import DatasetLoader
import yaml

with open('../config/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
loader = DatasetLoader('../../IO-VNBD-master', '../../Data_details/data/raw')
s_df, _ = loader.load_sequence(cfg['selected_sequence']['smartphone_file'])
ay = s_df['acc_y'].values


## 2. Causal vs Non-Causal Comparison
Verify that causal filtering operates without future-data cheating.

In [ ]:
causal_filt = apply_butterworth_causal(ay, cutoff_hz=1.5, fs=10.0, order=2)
offline_filt = apply_butterworth_offline(ay, cutoff_hz=1.5, fs=10.0, order=2)

plt.figure(figsize=(12, 4))
plt.plot(ay[1000:1300], color='#bdc3c7', label='Raw Accel (with Vibration)')
plt.plot(offline_filt[1000:1300], 'r--', label='Offline Zero-Phase (filtfilt)')
plt.plot(causal_filt[1000:1300], 'b-', label='Causal Streaming (lfilter)')
plt.title('Causal vs Non-Causal Filtering on Real Driving Maneuver')
plt.xlabel('Sample (100 ms per step)')
plt.ylabel('Forward Accel (m/s²)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 3. Streaming Stateful Filter
Process sample-by-sample with RealTimeCausalFilter.

In [ ]:
cf = RealTimeCausalFilter(cutoff_hz=1.5, fs=10.0, order=2)
stream_out = [cf.process_sample(val) for val in ay[:100]]
print(f'Processed {len(stream_out)} samples causally in real time.')
